# 第 4 堂　答案要有依據

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rthtsai/junior-lab/blob/main/w04.ipynb)

**怎麼用**：由上往下，一格一格按左邊的 ▶ 執行。

- 看到 **✏️** 的格子，要自己把 `____` 換成答案。沒填就按，會出現紅色錯誤，這是正常的。
- 卡住的順序：先看格子裡的提示 → 問同組 → 再舉手。
- 上週程式學會指出依據句。今天答案已經由老師寫好，程式要**檢查每一句答案有沒有照著依據寫**。

## 先做這一格（一開始就按）

下載語意模型要 1 到 3 分鐘，先讓它在背景跑。
如果跳出「要重新啟動工作階段」，按「重新啟動」再繼續，不用重按這格。

In [ ]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-m3")
print("語意模型準備好了")

---
## 第 1 節　題目、依據、答案

題組一有一個題目、一段依據、一份待查證的答案（六句）。
**先不要跑程式，用眼睛看**：哪幾句答案真的有依據？把編號記下來，填進表單第 1 題。

In [ ]:
import re

question = "為什麼打雷時，我們先看到閃電，才聽到雷聲？"
passage = "打雷時，閃電和雷聲其實是同時產生的。光的速度約為每秒 30 萬公里，幾乎一瞬間就到達我們的眼睛。聲音在空氣中的速度約為每秒 340 公尺，比光慢很多。所以我們總是先看到閃電，過了一會兒才聽到雷聲。閃電和雷聲相隔的時間越長，表示打雷的地方離我們越遠。聲音大約每 3 秒前進 1 公里，數一數秒數就能估計距離。"
answer = "光的速度約為每秒 30 萬公里。聲音在空氣中每秒約前進 340 公尺，比光慢很多。聲音在空氣中每秒約前進 3400 公尺，比光慢很多。因為閃電比雷聲先發生，所以我們先看到閃電。閃電和雷聲相隔的時間越長，代表打雷的地方離我們越遠。閃電和雷聲不是同時產生的。"

def split_sentences(text):
    return [s for s in text.split("。") if s]

print("題目：", question)
print()
print("【依據】")
for i, s in enumerate(split_sentences(passage), 1):
    print(f"  {i}. {s}")
print()
print("【待查證的答案】")
for i, s in enumerate(split_sentences(answer), 1):
    print(f"  {i}. {s}")

---
## 第 2 節　第一道檢查：詞彙覆蓋

把答案句每**兩個字**切成一組（叫詞塊），看有多少比例的詞塊，也出現在依據句裡。比例太低，代表答案句講了依據裡沒有的東西。

### ✏️ 1　覆蓋率的分母

`a` 是答案句的詞塊，`b` 是依據句的詞塊。覆蓋率是「答案句的詞塊有幾成出現在依據句」。

提示：分母是答案句的詞塊，還是依據句的？

In [ ]:
def chunks(text):
    text = re.sub(r"[，。、：；！？\s]", "", text)
    return {text[i:i+2] for i in range(len(text) - 1)}

def coverage(answer_sent, evidence_sent):
    a = chunks(answer_sent)
    b = chunks(evidence_sent)
    return len(a & b) / len(____)

print(round(coverage("光的速度約為每秒 30 萬公里。", "光的速度約為每秒 30 萬公里，幾乎一瞬間就到達我們的眼睛。"), 2))

---
## 第 3 節　第二道檢查：數字與否定詞

兩個字一組的比對看不出「340」和「3400」不同，也看不出有沒有「不」。所以另外檢查：
1. 答案句裡的每個數字，都要在依據段裡找得到。
2. 答案句有沒有否定詞（不、沒有、無法、不能），要和那句依據一樣。

### ✏️ 2　數字要找得到

提示：要檢查的是答案句的數字，變數叫什麼？

In [ ]:
NEG = ["不", "沒有", "無法", "不能"]

def numbers(text):
    return re.findall(r"\d+", text)

def has_neg(text):
    return any(w in text for w in NEG)

def numneg_ok(answer_sent, evidence_sent, passage_text):
    a_nums = numbers(answer_sent)
    passage_nums = numbers(passage_text)
    num_ok = all(n in passage_nums for n in ____)
    neg_ok = has_neg(answer_sent) == has_neg(evidence_sent)
    return num_ok and neg_ok

print(numneg_ok("聲音每秒前進 3400 公尺。", "聲音每秒前進 340 公尺。", "聲音每秒前進 340 公尺。"))

---
## 第 4 節　第三道檢查：語意，再三道合判

每一句答案，先用語意相似度找出**最像的那一句依據**，再做三道檢查。下面已經寫好：找最像的依據句。

`COV`、`TH` 是兩個門檻，`ON` 是現在開著的檢查，可以自己改。

### ✏️ 3　三道全過

`checks` 是三道檢查的結果，`ON` 是現在開著的檢查名單。只看名單上的檢查，全部通過才算有依據。

提示：`for k in ____`，要走過的是哪一份名單？

In [ ]:
COV = 0.4     # 覆蓋率門檻
TH = 0.87      # 語意相似度門檻
ON = ["覆蓋", "數字否定", "語意"]   # 現在開著的檢查

def extra_check(answer_sent, evidence_sent, passage_text):
    return True    # 作業才會用到，今天先不動

def check_answer(answer_text, passage_text, show=True):
    ev_sents = split_sentences(passage_text)
    ev_vecs = model.encode(ev_sents, normalize_embeddings=True)
    results = []
    for a in split_sentences(answer_text):
        v = model.encode([a], normalize_embeddings=True)
        sims = [float(x) for x in (ev_vecs @ v.T).ravel()]
        i = max(range(len(sims)), key=lambda k: sims[k])   # 最像的那句依據
        ev = ev_sents[i]
        cov = coverage(a, ev)
        checks = {"覆蓋": cov >= COV,
                  "數字否定": numneg_ok(a, ev, passage_text),
                  "語意": sims[i] >= TH,
                  "自訂": extra_check(a, ev, passage_text)}
        supported = all(checks[k] for k in ____)
        results.append({"answer": a, "evidence": ev, "cov": cov, "sim": sims[i],
                        "checks": checks, "supported": supported})
    if show:
        for n, r in enumerate(results, 1):
            ch = r["checks"]
            mark = "有依據" if r["supported"] else "沒依據"
            blocked = [k for k in ON if not ch[k]]
            print(f"{n}. {mark}  覆蓋 {r['cov']:.2f}  語意 {r['sim']:.3f}  擋下的檢查：{blocked or '無'}")
            print("    答案：", r["answer"])
            print("    最像的依據：", r["evidence"])
    return results

results = check_answer(answer, passage)

### ✏️ 4　有依據比例

**有依據比例 ＝ 通過的句數 ÷ 總句數**。（和上週的「依據命中率」不一樣：那是選對依據句，這個是答案句有沒有依據。）

提示：分母是哪一個數字？

In [ ]:
def support_ratio(results):
    return sum(r["supported"] for r in results) / ____

passed = sum(r["supported"] for r in results)
print("有依據比例", passed, "/", len(results), "=", round(support_ratio(results), 2))

---
## 實驗　拆掉一道檢查，會多放進誰？

1. 把上面 `ON` 那一行改成只剩兩道，例如 `ON = ["覆蓋", "語意"]`（拿掉數字否定），重新執行「三道全過」那一格，再執行「有依據比例」那一格。記下多放進哪幾句、比例變多少。
2. 改回三道，再試只拿掉「語意」。
3. 改回三道，把 `TH` 改成 0.70，再改成 0.95，各跑一次。
4. 最後把 `ON`、`TH` 都改回來。

每改一次，記下：有依據比例、多放進或被擋下哪一句。回表單。

## 照抄示範

如果把**整段依據原文**當成答案交上去，會怎麼樣？按執行看看。

In [ ]:
copy_results = check_answer(passage, passage, show=False)
print("整段依據當答案，有依據比例", sum(r["supported"] for r in copy_results), "/", len(copy_results))

### 想一想（回表單）

1. 哪一句**相似度很高，卻沒有依據**？它的相似度比哪一句有依據的句子還高？
2. 整段原文當答案，比例是多少？這算好答案嗎？

---
## 解鎖標準答案

標準答案要等點評之後才打開。老師會給你解鎖碼，填進下面的引號再按執行。

判斷正確率 ＝ 程式的「有／沒依據」和標準答案一致的句數 ÷ 總句數。

In [ ]:
import hashlib, base64, json
GOLD_BLOB = "wgVShDqekbeOb0JXNSXdVD/O0+xqrdgs+aCyIm7ZhvCICy7BXoqF9o5vTAs1cqEQKpDHg3uz2if127MibtmG8IgLLsEvhpT48iooHyEzoRAkzMfUB/TPeeG0oj1s04qLiAsuwC+GlfjyK1kTMD3dVUCJ"

def open_gold(code):
    key = hashlib.sha256(code.encode()).digest()
    raw = base64.b64decode(GOLD_BLOB)
    data = bytes(b ^ key[i % len(key)] for i, b in enumerate(raw))
    try:
        return json.loads(data.decode())
    except Exception:
        print("解鎖碼不對，或老師還沒公布")
        return None

GOLD = open_gold("")   # 把解鎖碼填進引號
print("標準答案", "已打開" if GOLD else "還沒打開")

---
## 應用任務　三個題組，算兩個數字

下面兩格已經寫好，**只要按執行**。每個題組六句答案，印出每句結果、有依據比例、判斷正確率。

In [ ]:
SETS = {
    "題組 1": {
        "q": "為什麼打雷時，我們先看到閃電，才聽到雷聲？",
        "passage": "打雷時，閃電和雷聲其實是同時產生的。光的速度約為每秒 30 萬公里，幾乎一瞬間就到達我們的眼睛。聲音在空氣中的速度約為每秒 340 公尺，比光慢很多。所以我們總是先看到閃電，過了一會兒才聽到雷聲。閃電和雷聲相隔的時間越長，表示打雷的地方離我們越遠。聲音大約每 3 秒前進 1 公里，數一數秒數就能估計距離。",
        "answer": "光的速度約為每秒 30 萬公里。聲音在空氣中每秒約前進 340 公尺，比光慢很多。聲音在空氣中每秒約前進 3400 公尺，比光慢很多。因為閃電比雷聲先發生，所以我們先看到閃電。閃電和雷聲相隔的時間越長，代表打雷的地方離我們越遠。閃電和雷聲不是同時產生的。"
    },
    "題組 2": {
        "q": "為什麼蘋果切開以後會變成褐色？",
        "passage": "蘋果切開後，果肉接觸到空氣，很快就會變成褐色。蘋果裡有一種叫多酚氧化酶的物質，遇到氧氣時，會讓果肉裡的多酚變成褐色的物質。所以切開的蘋果放得越久，變成褐色的地方就越多。檸檬汁是酸性的，又含有抗氧化的成分，可以減緩這種變色。把切好的蘋果泡在水裡，隔絕空氣，也能延緩變色。",
        "answer": "蘋果切開後，果肉接觸到空氣，就會慢慢變成褐色。果肉裡的鐵接觸到氧氣就會生鏽，所以蘋果變成褐色。多酚氧化酶遇到氧氣，會讓果肉裡的多酚變成褐色的物質。變成褐色的蘋果有毒，吃了會對身體造成傷害。把切好的蘋果泡在水裡，可以隔絕空氣，延緩變色。檸檬汁不能減緩蘋果變色。"
    },
    "題組 3": {
        "q": "為什麼吃糖容易蛀牙？",
        "passage": "吃進嘴裡的糖，會被口腔裡的細菌分解。細菌分解糖的時候，會產生酸。酸會慢慢溶解牙齒表面的琺瑯質，讓牙齒出現小洞。這些小洞越來越大，就是我們說的蛀牙。刷牙可以清除牙齒上的細菌和殘留的糖，減少酸的產生。",
        "answer": "口腔裡的細菌分解糖的時候，會產生酸。酸會慢慢溶解牙齒表面的琺瑯質，讓牙齒出現小洞。糖本身就是酸，酸會慢慢溶解牙齒表面的琺瑯質，讓牙齒出現小洞。刷牙可以清除牙齒上的細菌和殘留的糖，減少酸的產生。小洞越來越大以後，才會被稱為蛀牙。蛀牙是牙齒裡的蟲在咬出來的洞。"
    }
}

In [ ]:
def run_all(verbose=True):
    total_right = total_n = 0
    for name, S in SETS.items():
        res = check_answer(S["answer"], S["passage"], show=False)
        gold = GOLD[name]
        right = sum(int(r["supported"]) == g for r, g in zip(res, gold))
        passed = sum(r["supported"] for r in res)
        if verbose:
            print(name, "：", S["q"])
            for n, (r, g) in enumerate(zip(res, gold), 1):
                ok = "✔" if int(r["supported"]) == g else "✘"
                print(" ", ok, n, "有依據" if r["supported"] else "沒依據", f"覆蓋 {r['cov']:.2f} 語意 {r['sim']:.3f}", r["answer"][:24])
        print(name, "有依據比例", passed, "/", len(res), "　判斷正確率", right, "/", len(res))
        total_right += right; total_n += len(res)
    print("三題組合計判斷正確率", total_right, "/", total_n)
    return total_right, total_n

if GOLD:
    run_all()

### 要交的（填在表單）

1. 三個題組的**判斷正確率**、**有依據比例**各是多少？
2. 程式判錯的是哪一句？三道檢查為什麼都沒擋下它？
3. 把 `TH` 改成 0.93 再跑一次（下一格），合計判斷正確率變成多少？有沒有任何門檻能全對？

In [ ]:
if GOLD:
    for t in [0.70, 0.80, 0.87, 0.93]:
        TH = t
        right, n_all = run_all(verbose=False)
        print("TH =", t, "→ 判斷正確率", right, "/", n_all)
    TH = 0.87   # 記得改回來

---
## 下週前要交（回家做）

### 一　自己寫三句陷阱，再把它們擋下來

1. 在 `my_answer` 寫幾句答案（可以用題組一的依據，也可以自己出題），**其中三句是你故意寫來騙過程式的陷阱**，其他是有依據的。
2. 執行，記下哪幾句通過了、有依據比例。
3. 改規則（改 `extra_check`、`ON`、`COV` 或 `TH`）把陷阱擋下來，再執行一次，記下改完的有依據比例。

In [ ]:
my_passage = passage      # 也可以換成自己的依據段
my_answer = "在這裡寫你的答案，每句用句號結尾。"
_ = check_answer(my_answer, my_passage)
print("有依據比例", sum(r["supported"] for r in _), "/", len(_))

`extra_check` 是留給你的空位。例如你可以讓它檢查「近／遠」「快／慢」這類相反的詞有沒有跑掉。改好之後，要把 `"自訂"` 加進 `ON` 才會啟用。

### 二　檢查「AI 寫的答案」

下面是一份 AI 寫的答案（對題組一的依據）。用你的程式查證，找出**幻覺那一句**（依據裡沒有、甚至和依據不合的），用一句話說明：它為什麼被擋，或為什麼沒被擋。

In [ ]:
ai_answer = "閃電和雷聲其實是同時產生的。所以我們總是先看到閃電，才聽到雷聲。閃電和雷聲相隔的時間越長，打雷的地方離我們越遠。光的速度約為每秒 30 萬公里，所以閃電一定比雷聲先發生。"
_ = check_answer(ai_answer, passage)

### 三　存副本

**File（檔案）→ 在雲端硬碟中儲存副本**，存到自己的雲端硬碟。